# Real Drive-Time Isochrones using OSRM (Open Source)

Uses OSRM (Open Source Routing Machine) public API for real road network routing.
- **100% Free & Open Source**
- **No installation needed** - uses public OSRM demo server
- **Real routing** using actual road network
- **No API key required**

Note: The public server has rate limits. For high-volume production use, consider self-hosting OSRM.

## Parameters

In [ ]:
dbutils.widgets.text("catalog", "jdub_demo_aws")
dbutils.widgets.text("schema", "geospatial_site_selection")
dbutils.widgets.text("input_table", "bronze_rmc_retail_locations_grocery", "Input Locations")
dbutils.widgets.text("output_table", "silver_rmc_isochrones", "Output Table")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
input_table = dbutils.widgets.get("input_table")
output_table = dbutils.widgets.get("output_table")

# Use public OSRM demo server
OSRM_BASE_URL = "https://router.project-osrm.org"

## Setup - Test Connection to Public OSRM API

In [ ]:
import requests
import time

# Test connection to public OSRM server
print(f"Testing connection to {OSRM_BASE_URL}...")

try:
    # Test with a simple route in Boston area
    test_url = f"{OSRM_BASE_URL}/route/v1/driving/-71.0589,42.3601;-71.0603,42.3584"
    response = requests.get(test_url, timeout=10)
    
    if response.status_code == 200:
        data = response.json()
        if 'routes' in data:
            print("✓ Successfully connected to public OSRM server")
            print(f"  Server: {OSRM_BASE_URL}")
            print(f"  Test route duration: {data['routes'][0]['duration']:.1f} seconds")
        else:
            print("⚠ Connected but unexpected response format")
    else:
        print(f"⚠ Server returned status {response.status_code}")
        raise Exception(f"OSRM server not available: {response.status_code}")
        
except Exception as e:
    print(f"❌ Could not connect to OSRM server: {e}")
    print("\nNote: The public OSRM demo server may have rate limits.")
    print("If this fails, consider using Mapbox API or self-hosting OSRM.")
    raise

# Add a small delay to be respectful of the public server
time.sleep(1)

## Load Locations and Urbanicity

In [ ]:
from pyspark.sql.functions import col, expr, broadcast, lit

# Read locations
locations = spark.table(f"{catalog}.{schema}.{input_table}")

# Auto-detect columns
columns = locations.columns
id_col = next((c for c in columns if c in ['store_number', 'point_id', 'id', 'location_id']), columns[0])
lat_col = next((c for c in columns if c in ['latitude', 'lat', 'y']), None)
lon_col = next((c for c in columns if c in ['longitude', 'lon', 'lng', 'x']), None)

if not lat_col or not lon_col:
    raise ValueError(f"Cannot find lat/lon columns. Available: {columns}")

# Standardize columns
locations_std = locations.select(
    col(id_col).alias("location_id"),
    col(lat_col).alias("latitude"),
    col(lon_col).alias("longitude")
).filter(col("latitude").isNotNull() & col("longitude").isNotNull())

# Load H3 features for urbanicity
h3_features = spark.table(f"{catalog}.{schema}.silver_h3_features").select(
    col("h3_cell_id"),
    col("urbanicity_category")
)

# Add urbanicity
locations_with_urbanicity = (
    locations_std
    .withColumn("h3_cell", expr("h3_longlatash3string(longitude, latitude, 9)"))
    .join(
        broadcast(h3_features.withColumnRenamed("h3_cell_id", "h3_cell")),
        "h3_cell",
        "left"
    )
    .fillna({"urbanicity_category": "suburban"})
)

# Add drive times
locations_with_times = locations_with_urbanicity.withColumn(
    "drive_time_minutes",
    expr("""
        CASE
            WHEN urbanicity_category = 'urban' THEN 10
            WHEN urbanicity_category = 'suburban' THEN 20
            WHEN urbanicity_category = 'rural' THEN 30
            ELSE 20
        END
    """)
)

print(f"Loaded {locations_with_times.count()} locations")
display(locations_with_times.groupBy("urbanicity_category", "drive_time_minutes").count())

## Generate Isochrones using OSRM

In [ ]:
import requests
import json
from shapely.geometry import LineString, Polygon
from shapely.ops import unary_union
import math
import time

def get_osrm_isochrone(lon, lat, minutes, base_url=OSRM_BASE_URL):
    """
    Generate isochrone using OSRM API

    OSRM doesn't have native isochrone support, so we:
    1. Generate routes in multiple directions
    2. Find points at target drive time
    3. Create polygon from those points
    """

    # Generate points in a circle around location
    num_directions = 32  # More directions = smoother polygon
    distance_km = minutes * 1.0  # Rough estimate: 60 km/h average speed

    points_at_time = []

    for i in range(num_directions):
        angle = (2 * math.pi * i) / num_directions

        # Calculate destination point (rough approximation)
        lat_offset = (distance_km / 111.32) * math.cos(angle)
        lon_offset = (distance_km / (111.32 * math.cos(math.radians(lat)))) * math.sin(angle)

        dest_lon = lon + lon_offset
        dest_lat = lat + lat_offset

        # Get route from OSRM
        try:
            url = f"{base_url}/route/v1/driving/{lon},{lat};{dest_lon},{dest_lat}"
            params = {
                'overview': 'full',
                'geometries': 'geojson'
            }

            response = requests.get(url, params=params, timeout=10)

            if response.status_code == 200:
                data = response.json()

                if 'routes' in data and len(data['routes']) > 0:
                    route = data['routes'][0]
                    route_duration = route['duration'] / 60  # Convert to minutes
                    coordinates = route['geometry']['coordinates']

                    # Find point closest to target time
                    if route_duration > 0:
                        target_ratio = minutes / route_duration
                        if target_ratio <= 1:
                            target_idx = int(len(coordinates) * target_ratio)
                            target_idx = min(target_idx, len(coordinates) - 1)
                            points_at_time.append(coordinates[target_idx])
            
            # Small delay to respect rate limits on public server
            time.sleep(0.05)
            
        except Exception as e:
            # Silently continue on errors (some routes may fail)
            continue

    # Create polygon from points
    if len(points_at_time) >= 3:
        try:
            polygon = Polygon(points_at_time)
            polygon = polygon.convex_hull

            # Convert to WKT
            coords_str = ', '.join([f"{lon} {lat}" for lon, lat in polygon.exterior.coords])
            return f"POLYGON (({coords_str}))"
        except:
            return None

    return None

In [ ]:
# Test single isochrone
test_location = locations_with_times.first()
print(f"Testing isochrone generation for: {test_location.location_id}")
print(f"Location: ({test_location.latitude}, {test_location.longitude})")
print(f"Drive time: {test_location.drive_time_minutes} minutes")
print(f"\nGenerating isochrone with {32} route samples...")

wkt = get_osrm_isochrone(test_location.longitude, test_location.latitude, test_location.drive_time_minutes)

if wkt:
    print("\n✓ Test isochrone generated successfully!")
    print(f"  WKT polygon length: {len(wkt)} characters")
    print(f"  Using public OSRM server: {OSRM_BASE_URL}")
else:
    print("\n⚠ Test failed - isochrone could not be generated")
    print("  This may be due to rate limits or connectivity issues")

## Generate All Isochrones

In [ ]:
from pyspark.sql import Row
import time

location_rows = locations_with_times.collect()
print(f"Generating isochrones for {len(location_rows)} locations...")
print(f"Using public OSRM server: {OSRM_BASE_URL}")
print(f"Estimated time: ~{len(location_rows) * 2} seconds (with rate limiting)")
print("")

results = []
start_time = time.time()

for i, row in enumerate(location_rows):
    if i % 5 == 0:
        elapsed = time.time() - start_time
        if i > 0:
            avg_time = elapsed / i
            remaining = (len(location_rows) - i) * avg_time
            print(f"Progress: {i}/{len(location_rows)} ({i/len(location_rows)*100:.1f}%) - "
                  f"Elapsed: {elapsed:.1f}s - ETA: {remaining:.1f}s")

    wkt = get_osrm_isochrone(row.longitude, row.latitude, row.drive_time_minutes)

    if wkt:
        results.append(Row(
            location_id=row.location_id,
            latitude=row.latitude,
            longitude=row.longitude,
            urbanicity_category=row.urbanicity_category,
            drive_time_minutes=row.drive_time_minutes,
            geometry_wkt=wkt
        ))

total_time = time.time() - start_time
print(f"\n✅ Generated {len(results)}/{len(location_rows)} isochrones successfully")
print(f"   Total time: {total_time:.1f} seconds")
print(f"   Average: {total_time/len(location_rows):.2f} seconds per location")

## Save to Delta

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
from pyspark.sql.functions import current_timestamp

# Create DataFrame
isochrone_schema = StructType([
    StructField("location_id", StringType(), False),
    StructField("latitude", DoubleType(), False),
    StructField("longitude", DoubleType(), False),
    StructField("urbanicity_category", StringType(), True),
    StructField("drive_time_minutes", IntegerType(), False),
    StructField("geometry_wkt", StringType(), False)
])

isochrones_df = spark.createDataFrame(results, schema=isochrone_schema)

# Convert to geometry and add metadata
isochrones_final = (
    isochrones_df
    .withColumn("geometry", expr("ST_GeomFromText(geometry_wkt, 4326)"))
    .withColumn("area_sqkm", expr("ST_Area(geometry) / 1000000"))
    .withColumn("created_timestamp", current_timestamp())
    .withColumn("routing_provider", lit("osrm"))
    .drop("geometry_wkt")
)

# Save
output_table_name = f"{catalog}.{schema}.{output_table}"

(
    isochrones_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table_name)
)

print(f"✅ Saved {len(results)} isochrones to {output_table_name}")

## Summary

In [ ]:
display(spark.sql(f"""
    SELECT
        urbanicity_category,
        drive_time_minutes,
        COUNT(*) as count,
        ROUND(AVG(area_sqkm), 2) as avg_area_sqkm
    FROM {output_table_name}
    GROUP BY urbanicity_category, drive_time_minutes
    ORDER BY urbanicity_category
"""))